Partition and move criteo data to s3

In [ ]:
from datasets import load_dataset
import os

train_file_path = "../../../datasets/criteo_attribution_dataset"

full = load_dataset(
    'csv',
    data_files=os.path.join(train_file_path, "criteo_attribution_dataset.tsv"),
    delimiter='\t'
)
full

In [ ]:
import s3fs
import polars as pl
from datetime import datetime, timezone

# import math

S3_BUCKET = "<BUCKET-NAME>"
S3_PREFIX = "criteo/unprocessed"
# N_SHARDS = 20

fs = s3fs.S3FileSystem()

dataset = full["train"]
# shard_size = math.ceil(len(dataset) / N_SHARDS)

# for i in range(N_SHARDS):
#     start = i * shard_size
#     end = min(start + shard_size, len(dataset))
#     shard = dataset.select(range(start, end))
#     s3_path = f"s3://{S3_BUCKET}/{S3_PREFIX}/part-{i:05d}.parquet"
#     with fs.open(s3_path, "wb") as f:
#         shard.to_parquet(f)
#     print(f"[{i+1}/{N_SHARDS}] written {len(shard)} rows → {s3_path}")

now = datetime.now(timezone.utc).replace(tzinfo=None)
# fast vectorized grouping
df = dataset.to_polars()
# Add a day column derived from the timestamp
df = df.with_columns(
    (pl.col("timestamp") // 86400).alias("day")
)

df = df.with_columns(
    (pl.lit(now) + pl.duration(seconds=pl.col("timestamp") - df["timestamp"].max()))
    # .dt.strftime("%Y-%m-%d %H:%M:%S")
    .dt.replace_time_zone(None)
    .cast(pl.Datetime("us"))
    .alias("event_timestamp")
)

partitions = df.partition_by("day", as_dict=True)

for day, part in partitions.items():
    s3_path = f"s3://{S3_BUCKET}/{S3_PREFIX}/day={day[0]}/data.parquet"
    with fs.open(s3_path, "wb") as f:
        part.drop("day").write_parquet(
            f,
            use_pyarrow=True,
            pyarrow_options={
                "coerce_timestamps": "us",
                "allow_truncated_timestamps": True,
            }
        )
    print(f"day={day[0]}: {len(part)} rows → {s3_path}")

In [18]:
df.schema

Schema([('timestamp', Int64),
        ('uid', Int64),
        ('campaign', Int64),
        ('conversion', Int64),
        ('conversion_timestamp', Int64),
        ('conversion_id', Int64),
        ('attribution', Int64),
        ('click', Int64),
        ('click_pos', Int64),
        ('click_nb', Int64),
        ('cost', Float64),
        ('cpo', Float64),
        ('time_since_last_click', Int64),
        ('cat1', Int64),
        ('cat2', Int64),
        ('cat3', Int64),
        ('cat4', Int64),
        ('cat5', Int64),
        ('cat6', Int64),
        ('cat7', Int64),
        ('cat8', Int64),
        ('cat9', Int64),
        ('day', Int64),
        ('event_timestamp', Datetime(time_unit='us', time_zone=None))])